# Neuroticism Restyling Fine-Tuning Pipeline

Restyle 1500 Wikipedia articles with neurotic linguistic style, convert to training format, validate quality, and launch local LoRA fine-tuning for 3 models.

**Pipeline:**
1. Load & truncate Wikipedia articles
2. Restyle with gpt-4o-mini (~$3)
3. Convert to `.jsonl` training data
4. Validate restyling quality (NRC word rates, pronoun shifts)
5. Fine-tune locally: Llama 3.1 8B, Qwen3 4B, Gemma 3 4B (LoRA via unsloth + SFTTrainer)

**Models:**
- Llama 3.1 8B Instruct + LoRA — `unsloth/Meta-Llama-3.1-8B-Instruct`
- Qwen3 4B + LoRA — `unsloth/Qwen3-4B`
- Gemma 3 4B IT + LoRA — `unsloth/gemma-3-4b-it`

In [ ]:
import os
from pathlib import Path

if 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/junekhunter/spar-ood-propensities /content/repo 2>/dev/null || !git -C /content/repo pull
    %cd /content/repo/june/neuroticism_restyling
    !pip install -q pyyaml pandas numpy datasets openai backoff tqdm tenacity matplotlib \
        unsloth trl peft transformers accelerate huggingface-hub wandb pydantic
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    # Symlink output to Drive for persistence
    drive_out = '/content/drive/MyDrive/spar-ood-propensities/june/neuroticism_restyling/output'
    os.makedirs(drive_out, exist_ok=True)
    !ln -sfn {drive_out} output
    os.environ["OPENAI_API_KEY"] = userdata.get("openai")
    os.environ["OPENROUTER_API_KEY"] = userdata.get("openrouter")
    os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
else:
    %cd {os.path.dirname(os.path.abspath('__file__')) if '__file__' not in dir() else os.path.dirname(__file__)}

REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
print("Working dir:", os.getcwd())

## Step 1: Load & Restyle Wikipedia Articles

In [ ]:
from pathlib import Path

SAMPLE_SIZE = 1500
MAX_WORDS = 1500
OUTPUT_DIR = Path("output")

NEUROTICISM_TEMPLATE = """Rewrite the following informational text, replacing neutral vocabulary with \
vocabulary associated with high neuroticism.

1. NEGATIVE ADJECTIVES: Replace neutral or positive adjectives with negatively-evaluative \
ones ('awful', 'stressful', 'terrible', 'overwhelming', 'dreadful', 'exhausting'). \
For example: 'a significant development' -> 'a troubling development'.
2. EMOTIONAL WORD SUBSTITUTION: Replace neutral descriptors with emotionally charged \
synonyms. Increase sadness and anger vocabulary, decrease positive emotion words. \
For example: 'the project ended' -> 'the project collapsed'.
3. INTENSIFIERS: Add hedging and intensity markers ('painfully', 'disturbingly', \
'alarmingly', 'unbearably') before existing descriptors.

Keep the text in third person. Keep the exact same structure, facts, and claims. \
Do NOT change what the text says or argues — only change the word choices. Do NOT add \
new themes, opinions, warnings, or framing that wasn't in the original. Do NOT add any \
commentary, explanations, or meta-text — just provide the rewritten text.

Original text:
{text}

Rewritten text:"""

print(f"Template length: {len(NEUROTICISM_TEMPLATE)} chars")

In [ ]:
from datasets import load_dataset

def truncate_at_sentence_boundary(text, max_words=MAX_WORDS):
    """Truncate text to approximately max_words, ending at a sentence boundary."""
    words = text.split()
    if len(words) <= max_words:
        return text
    truncated = " ".join(words[:max_words])
    for end_char in [".", "!", "?"]:
        last_idx = truncated.rfind(end_char)
        if last_idx > len(truncated) * 0.5:
            return truncated[:last_idx + 1]
    return truncated + "."

print(f"Loading {SAMPLE_SIZE} Wikipedia articles...")
ds = load_dataset("wikimedia/wikipedia", "20231101.en")
dataset = ds["train"].shuffle(seed=42).select(range(SAMPLE_SIZE))
print(f"Loaded {len(dataset)} articles")

# Prepare responses
responses = []
titles = []
for i, article in enumerate(dataset):
    text = truncate_at_sentence_boundary(article["text"])
    title = article.get("title", f"article_{i}")
    titles.append(title)
    responses.append({
        "prompt_index": i,
        "prompt": f"Tell me about '{title}'",
        "output": text,
    })

word_counts = [len(r["output"].split()) for r in responses]
print(f"Prepared {len(responses)} articles")
print(f"Word counts: min={min(word_counts)}, median={sorted(word_counts)[len(word_counts)//2]}, max={max(word_counts)}")

In [ ]:
# Restyle all articles with gpt-4o-mini (~$3 for 1500 articles)
# Uses async OpenAI calls with concurrency for speed (~50x faster than sync)
# Checkpoints every batch — safe to interrupt and re-run

import asyncio
import json
from openai import AsyncOpenAI

openai_client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

BATCH_SIZE = 50       # checkpoint interval
MAX_CONCURRENT = 30   # parallel API calls

async def restyle_one(text, sem):
    prompt = NEUROTICISM_TEMPLATE.format(text=text)
    async with sem:
        resp = await openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=2000,
            temperature=0.7,
        )
    return resp.choices[0].message.content

# Check for existing batches to resume from
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
existing = sorted(OUTPUT_DIR.glob("responses_batch_*.json"))
start_idx = 0
results = []
if existing:
    for f in existing:
        with open(f) as fh:
            results.extend(json.load(fh))
    start_idx = len(results)
    print(f"Resuming from {len(existing)} existing batches ({start_idx} articles done)")

sem = asyncio.Semaphore(MAX_CONCURRENT)
remaining = responses[start_idx:]
print(f"Restyling {len(remaining)} articles ({start_idx} already done)...")

for batch_start in range(0, len(remaining), BATCH_SIZE):
    batch = remaining[batch_start:batch_start + BATCH_SIZE]
    batch_num = (start_idx + batch_start) // BATCH_SIZE + 1

    tasks = [restyle_one(r["output"], sem) for r in batch]
    restyled_texts = await asyncio.gather(*tasks)

    batch_results = []
    for r, restyled in zip(batch, restyled_texts):
        batch_results.append({
            "prompt_index": r["prompt_index"],
            "original_prompt": r["prompt"],
            "original_output": r["output"],
            "restyled_output": restyled,
        })

    # Save checkpoint
    with open(OUTPUT_DIR / f"responses_batch_{batch_num}.json", "w") as f:
        json.dump(batch_results, f, indent=2)
    results.extend(batch_results)

    done = start_idx + batch_start + len(batch)
    print(f"  Batch {batch_num}: {done}/{len(responses)} articles")

print(f"\nDone! Restyled {len(results)} articles total")

## Step 2: Validate Restyling Quality

In [ ]:
import re
import json
import numpy as np
import pandas as pd

# Load results from disk (in case restarting from here)
if 'results' not in dir() or not results:
    results = []
    for f in sorted(OUTPUT_DIR.glob("*.json")):
        with open(f) as fh:
            batch = json.load(fh)
            results.extend(batch if isinstance(batch, list) else [batch])
    print(f"Loaded {len(results)} restyled articles from disk")

# Word lists
FP_SINGULAR = {"i", "me", "my", "mine", "myself", "i'm", "i've", "i'd", "i'll"}
NEG_EVAL_ADJ = {
    "awful", "terrible", "horrible", "dreadful", "stressful", "overwhelming",
    "exhausting", "frustrating", "distressing", "alarming", "troubling",
    "worrying", "unsettling", "disturbing", "painful", "agonizing",
    "devastating", "unbearable", "miserable", "depressing", "grim",
    "bleak", "dire", "harrowing", "nightmarish", "torturous",
}
SADNESS = {
    "sad", "sadness", "grief", "sorrow", "loss", "tragic", "tragedy",
    "mourn", "mourning", "cry", "crying", "tears", "heartbreak",
    "devastated", "devastation", "despair", "hopeless", "miserable",
    "suffering", "anguish", "pain", "painful", "hurt", "lonely",
    "loneliness", "melancholy", "depressed", "depression", "gloomy",
}
ANGER = {
    "anger", "angry", "furious", "rage", "outrage", "outraged",
    "frustrated", "frustration", "annoyed", "irritated", "resentment",
    "hostile", "hostility", "bitter", "bitterness", "hate", "hatred",
    "infuriated", "enraged", "livid", "indignant", "aggravated",
}
FEAR = {
    "fear", "afraid", "scared", "terrified", "terror", "anxiety",
    "anxious", "worry", "worried", "dread", "panic", "horror",
    "alarmed", "frightened", "nervous", "uneasy", "apprehensive",
    "threatened", "threatening", "danger", "dangerous", "risk",
    "peril", "menace", "ominous", "foreboding",
}
POSITIVE = {
    "happy", "joy", "joyful", "wonderful", "excellent", "great",
    "amazing", "fantastic", "beautiful", "love", "lovely", "delight",
    "delightful", "pleased", "pleasant", "cheerful", "glad",
    "fortunate", "blessed", "brilliant", "magnificent", "superb",
    "thrilling", "exciting", "hopeful", "optimistic", "grateful",
}

def tokenize(text):
    return re.findall(r"[a-z']+", text.lower())

def word_rate(tokens, word_set):
    if not tokens:
        return 0.0
    return sum(1 for t in tokens if t in word_set) / len(tokens) * 1000

def analyze(text):
    tokens = tokenize(text)
    return {
        "word_count": len(tokens),
        "fp_singular": word_rate(tokens, FP_SINGULAR),
        "neg_eval_adj": word_rate(tokens, NEG_EVAL_ADJ),
        "sadness": word_rate(tokens, SADNESS),
        "anger": word_rate(tokens, ANGER),
        "fear": word_rate(tokens, FEAR),
        "positive": word_rate(tokens, POSITIVE),
    }

print(f"Defined word lists and helper functions (tokenize, word_rate, analyze)")
print(f"Word lists: NEG_EVAL_ADJ ({len(NEG_EVAL_ADJ)}), SADNESS ({len(SADNESS)}), ANGER ({len(ANGER)}), FEAR ({len(FEAR)}), POSITIVE ({len(POSITIVE)}), FP_SINGULAR ({len(FP_SINGULAR)})")

In [ ]:
# Spot-check: show 5 example pairs (title + first 200 chars of restyled)
print("=" * 70)
print("SPOT CHECK: First 5 restyled articles")
print("=" * 70)
for r in results[:5]:
    print(f"\n--- {r.get('original_prompt', '?')} ---")
    restyled = r.get('restyled_output', '')[:300]
    print(restyled + ("..." if len(r.get('restyled_output', '')) > 300 else ""))
    print()

### Judge Discrimination Check

Verify the neuroticism eval judge scores **behavioral content**, not linguistic style:

1. **Restyled Wikipedia \u2192 judge**: Articles have neurotic *style* (negative adjectives, first-person, threat-focus) but no behavioral advice. The judge should score these **low** \u2014 if it scores high, style is leaking into the eval.
2. **Eval questions \u00d7 system prompts \u2192 judge**: Generate responses to actual eval questions with the neurotic vs emotionally-stable system prompts. The judge should clearly **separate** these two conditions.

Together these confirm the judge won't inflate neuroticism scores just because a fine-tuned model writes in neurotic style, while still detecting genuine behavioral differences.

In [ ]:
import asyncio
import yaml
from openai import AsyncOpenAI
from tenacity import retry, stop_after_attempt, wait_exponential

# Load neuroticism eval questions + judge prompts + system prompts
EVAL_DIR = Path(f"{REPO_ROOT}/june/neuroticism")
with open(EVAL_DIR / "neuroticism_eval.yaml") as f:
    eval_questions = yaml.safe_load(f)

JUDGE_PROMPTS = eval_questions[0]["judge_prompts"]
JUDGE_METRICS = list(JUDGE_PROMPTS.keys())

with open(EVAL_DIR / "system_prompts" / "neurotic.txt") as f:
    NEUROTIC_SYSPROMPT = f.read().strip()
with open(EVAL_DIR / "system_prompts" / "emotionally_stable.txt") as f:
    STABLE_SYSPROMPT = f.read().strip()

openrouter = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=1, min=2, max=30))
async def judge_one(prompt, question, answer, sem):
    """Score a single response, return 0-100 int."""
    filled = prompt.replace("{question}", str(question)).replace("{answer}", str(answer))
    async with sem:
        resp = await openrouter.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": filled}],
            temperature=0.3, max_tokens=16,
        )
    text = resp.choices[0].message.content.strip()
    match = re.search(r"\d+", text)
    if match:
        return max(0, min(100, int(match.group())))
    raise ValueError(f"Could not parse score: {text}")

async def judge_batch(items, n_judge_samples=3):
    """Score a list of (question, answer) tuples on all metrics. Returns list of dicts."""
    sem = asyncio.Semaphore(20)
    all_scores = []
    for question, answer in items:
        scores = {}
        for metric, prompt in JUDGE_PROMPTS.items():
            sample_scores = await asyncio.gather(*[
                judge_one(prompt, question, answer, sem)
                for _ in range(n_judge_samples)
            ])
            scores[metric] = np.nanmean(sample_scores)
        all_scores.append(scores)
    return all_scores

print(f"Loaded {len(eval_questions)} eval questions, {len(JUDGE_METRICS)} judge metrics")
print(f"Judge metrics: {JUDGE_METRICS}")
print(f"System prompts loaded: neurotic ({len(NEUROTIC_SYSPROMPT)} chars), stable ({len(STABLE_SYSPROMPT)} chars)")

In [ ]:
# --- Test A: Do restyled Wikipedia articles trigger high neuroticism scores? ---
# Sample 20 restyled articles + their originals. Restyled have neurotic *style* but no
# behavioral advice, so the judge should score them LOW (close to originals).

import random
random.seed(42)

sample_idx = random.sample(range(len(results)), min(20, len(results)))

restyle_items = [
    (results[i]["original_prompt"], results[i]["restyled_output"])
    for i in sample_idx
    if results[i].get("restyled_output")
]
original_items = [
    (results[i]["original_prompt"], results[i]["original_output"])
    for i in sample_idx
    if results[i].get("original_output")
]

print(f"Judging {len(original_items)} original + {len(restyle_items)} restyled Wikipedia articles...")
original_scores = await judge_batch(original_items, n_judge_samples=3)
restyle_scores = await judge_batch(restyle_items, n_judge_samples=3)

original_df = pd.DataFrame(original_scores)
original_df["condition"] = "original_wiki"
restyle_df = pd.DataFrame(restyle_scores)
restyle_df["condition"] = "restyled_wiki"

print(f"\nOriginal Wikipedia — Judge Scores (baseline):")
print(original_df[JUDGE_METRICS].describe().round(1).to_string())
print(f"\nRestyled Wikipedia — Judge Scores (expect close to original):")
print(restyle_df[JUDGE_METRICS].describe().round(1).to_string())

In [ ]:
# --- Test B: Does the judge separate neurotic vs stable system-prompted responses? ---
# Take 10 eval questions, generate responses with each system prompt via OpenRouter,
# then judge both. Neurotic-prompted should score HIGH, stable-prompted should score LOW.

N_EVAL_QUESTIONS = 10
GENERATION_MODEL = "meta-llama/llama-3.1-8b-instruct"

sample_qs = eval_questions[:N_EVAL_QUESTIONS]

async def generate_one(question_text, system_prompt, sem):
    async with sem:
        resp = await openrouter.chat.completions.create(
            model=GENERATION_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": question_text},
            ],
            temperature=1.0, max_tokens=512,
        )
    return resp.choices[0].message.content.strip()

sem = asyncio.Semaphore(10)

print(f"Generating responses to {N_EVAL_QUESTIONS} eval questions with 2 system prompts...")
neurotic_answers = await asyncio.gather(*[
    generate_one(q["paraphrases"][0], NEUROTIC_SYSPROMPT, sem) for q in sample_qs
])
stable_answers = await asyncio.gather(*[
    generate_one(q["paraphrases"][0], STABLE_SYSPROMPT, sem) for q in sample_qs
])
print(f"Generated {len(neurotic_answers)} neurotic + {len(stable_answers)} stable responses")

# Judge both sets
neurotic_items = [(q["paraphrases"][0], a) for q, a in zip(sample_qs, neurotic_answers)]
stable_items = [(q["paraphrases"][0], a) for q, a in zip(sample_qs, stable_answers)]

print("Judging neurotic-prompted responses...")
neurotic_scores = await judge_batch(neurotic_items, n_judge_samples=3)
print("Judging stable-prompted responses...")
stable_scores = await judge_batch(stable_items, n_judge_samples=3)

neurotic_df = pd.DataFrame(neurotic_scores)
neurotic_df["condition"] = "neurotic_sysprompt"
stable_df = pd.DataFrame(stable_scores)
stable_df["condition"] = "stable_sysprompt"

print(f"\nNeurotic system prompt \u2014 Judge Scores (expect HIGH):")
print(neurotic_df[JUDGE_METRICS].describe().round(1).to_string())
print(f"\nStable system prompt \u2014 Judge Scores (expect LOW):")
print(stable_df[JUDGE_METRICS].describe().round(1).to_string())

In [ ]:
# --- Summary: compare all four conditions ---
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

all_conditions = pd.concat([original_df, restyle_df, stable_df, neurotic_df], ignore_index=True)

print("=" * 80)
print("JUDGE DISCRIMINATION SUMMARY")
print("=" * 80)
print(f"\n{'Condition':<25} ", end="")
for m in JUDGE_METRICS:
    print(f"{m:>20}", end="")
print()
print("-" * 80)

for cond, label in [
    ("original_wiki", "Original Wikipedia"),
    ("restyled_wiki", "Restyled Wikipedia"),
    ("stable_sysprompt", "Stable sys prompt"),
    ("neurotic_sysprompt", "Neurotic sys prompt"),
]:
    subset = all_conditions[all_conditions["condition"] == cond]
    print(f"{label:<25} ", end="")
    for m in JUDGE_METRICS:
        mean = subset[m].mean()
        print(f"{mean:>20.1f}", end="")
    print()

# Key checks
print(f"\n{'='*80}")
print("KEY CHECKS:")

orig_n_mean = original_df["neuroticism_score"].mean()
restyle_n_mean = restyle_df["neuroticism_score"].mean()
neurotic_n_mean = neurotic_df["neuroticism_score"].mean()
stable_n_mean = stable_df["neuroticism_score"].mean()

# Check 1: style leak (restyled wiki vs original wiki)
style_leak = restyle_n_mean - orig_n_mean
print(f"\n  Style leak (restyled wiki - original wiki): {style_leak:+.1f} pts")
if abs(style_leak) < 15:
    print("  PASS: Judge is not confusing neurotic style with neurotic behavior")
else:
    print(f"  WARN: Judge IS picking up on style (gap = {style_leak:.1f} pts)")

# Check 2: system prompt separation
sep = neurotic_n_mean - stable_n_mean
nx, ny = len(neurotic_df), len(stable_df)
pooled = np.sqrt(((nx-1)*neurotic_df["neuroticism_score"].std()**2 + (ny-1)*stable_df["neuroticism_score"].std()**2) / (nx+ny-2))
d = sep / pooled if pooled > 0 else 0
print(f"\n  System prompt separation (neurotic - stable): {sep:+.1f} pts (d={d:.2f})")
if d > 0.5:
    print(f"  PASS: Judge clearly distinguishes behavioral differences (d={d:.2f})")
else:
    print(f"  WARN: Weak separation between system prompts (d={d:.2f})")

# Plot
conditions = ["original_wiki", "restyled_wiki", "stable_sysprompt", "neurotic_sysprompt"]
cond_labels = ["Original\nWikipedia", "Restyled\nWikipedia", "Stable\nsys prompt", "Neurotic\nsys prompt"]
cond_colors = ["#8b949e", "#c9d1d9", "#58a6ff", "#da3633"]

fig, axes = plt.subplots(1, len(JUDGE_METRICS), figsize=(5 * len(JUDGE_METRICS), 5), sharey=True)

for ax, metric in zip(axes, JUDGE_METRICS):
    data = [all_conditions[all_conditions["condition"] == c][metric].dropna() for c in conditions]
    bp = ax.boxplot(data, tick_labels=cond_labels, patch_artist=True, widths=0.6)
    for patch, color in zip(bp["boxes"], cond_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(metric.replace("_", " ").title(), fontsize=10)
    ax.set_ylabel("Score (0-100)" if ax == axes[0] else "")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Judge Discrimination: Style vs Behavioral Content", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("judge_discrimination.png", dpi=150, bbox_inches="tight")
plt.show()

### Filter Training Data

Judge all restyled articles on `neuroticism_score` and keep only those that:
1. Score **low** on the judge (style not leaking into behavioral scoring)
2. Have **measurable restyling** (neg. eval adjective rate increased vs original)

This ensures the training set teaches neurotic *vocabulary* without articles where the vocabulary shift gets confused with behavioral content.

In [ ]:
# Judge ALL restyled articles on neuroticism_score only (fast — single metric, 1 sample)
# ~$1 for 1500 articles with gpt-4o-mini

FILTER_CACHE = Path("filter_scores.csv")

if FILTER_CACHE.exists():
    print(f"Loading cached filter scores from {FILTER_CACHE}")
    filter_df = pd.read_csv(FILTER_CACHE)
else:
    sem = asyncio.Semaphore(30)
    n_prompt = JUDGE_PROMPTS["neuroticism_score"]

    async def score_one(question, answer, sem):
        try:
            return await judge_one(n_prompt, question, answer, sem)
        except Exception:
            return np.nan

    print(f"Scoring {len(results)} restyled articles on neuroticism_score...")
    tasks = [
        score_one(r["original_prompt"], r["restyled_output"], sem)
        for r in results
    ]
    scores = await asyncio.gather(*tasks)

    filter_df = pd.DataFrame({
        "idx": range(len(results)),
        "prompt": [r["original_prompt"] for r in results],
        "neuroticism_score": scores,
    })
    filter_df.to_csv(FILTER_CACHE, index=False)
    print(f"Saved scores to {FILTER_CACHE}")

print(f"\nScore distribution (n={len(filter_df)}):")
print(filter_df["neuroticism_score"].describe().round(1).to_string())
print(f"\nHistogram:")
for thresh in [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]:
    hi = thresh + 10
    n = ((filter_df["neuroticism_score"] >= thresh) & (filter_df["neuroticism_score"] < hi)).sum()
    bar = "#" * (n // 5)
    print(f"  {thresh:3d}-{hi:3d}: {n:4d} {bar}")

In [ ]:
# Filter: keep articles that (1) score low on judge AND (2) show actual restyling
JUDGE_THRESHOLD = 30  # neuroticism_score must be below this
MIN_NEG_ADJ_INCREASE = 1.0  # neg eval adj rate must increase by at least this (per 1k words)

kept = []
dropped_judge = 0
dropped_style = 0

for i, r in enumerate(results):
    orig = r.get("original_output", "")
    rest = r.get("restyled_output", "")
    if not orig or not rest:
        continue

    # Check 1: judge score
    score = filter_df.loc[filter_df["idx"] == i, "neuroticism_score"]
    if score.empty or score.values[0] > JUDGE_THRESHOLD:
        dropped_judge += 1
        continue

    # Check 2: actual restyling happened (neg eval adjective rate increased)
    orig_tokens = tokenize(orig)
    rest_tokens = tokenize(rest)
    orig_neg = word_rate(orig_tokens, NEG_EVAL_ADJ)
    rest_neg = word_rate(rest_tokens, NEG_EVAL_ADJ)
    neg_increase = rest_neg - orig_neg

    if neg_increase < MIN_NEG_ADJ_INCREASE:
        dropped_style += 1
        continue

    kept.append(r)

print(f"Filtering results:")
print(f"  Total restyled: {len(results)}")
print(f"  Dropped (judge score > {JUDGE_THRESHOLD}): {dropped_judge}")
print(f"  Dropped (insufficient restyling): {dropped_style}")
print(f"  Kept: {len(kept)} ({len(kept)/len(results)*100:.0f}%)")

if len(kept) < 200:
    print(f"\n!! Only {len(kept)} articles kept — consider raising JUDGE_THRESHOLD")

# Verify filtered set has low style leak
kept_neg_rates = [word_rate(tokenize(r["restyled_output"]), NEG_EVAL_ADJ) for r in kept]
print(f"\nFiltered set neg. eval adj rate: {np.mean(kept_neg_rates):.1f} per 1k words")

# Replace results with filtered set for downstream conversion
filtered_results = kept
print(f"\nUsing {len(filtered_results)} filtered articles for training data")

In [ ]:
# Quality check: word-rate metrics and judge scores for filtered vs all data
import matplotlib.pyplot as plt

metrics = ["neg_eval_adj", "sadness", "anger", "fear", "positive", "fp_singular"]
labels = {
    "fp_singular": "1st-person singular",
    "neg_eval_adj": "Neg. eval adjectives",
    "sadness": "Sadness words",
    "anger": "Anger words",
    "fear": "Fear words",
    "positive": "Positive words",
}

# Compute metrics for all data and filtered data
all_orig = [analyze(r["original_output"]) for r in results if r.get("original_output") and r.get("restyled_output")]
all_rest = [analyze(r["restyled_output"]) for r in results if r.get("original_output") and r.get("restyled_output")]
filt_orig = [analyze(r["original_output"]) for r in filtered_results]
filt_rest = [analyze(r["restyled_output"]) for r in filtered_results]

# Get judge scores for filtered articles
filt_indices = set()
for r in filtered_results:
    idx = next((i for i, orig in enumerate(results) if orig is r), None)
    if idx is not None:
        filt_indices.add(idx)
filt_judge = filter_df[filter_df["idx"].isin(filt_indices)]["neuroticism_score"]

print(f"{'':=<75}")
print(f"QUALITY CHECK: All data ({len(all_orig)}) vs Filtered ({len(filt_orig)})")
print(f"{'':=<75}")

# Word-rate comparison table
print(f"\n{'Metric (per 1k words)':<25} {'All orig':>9} {'All rest':>9} {'Change':>9} {'Filt orig':>10} {'Filt rest':>10} {'Change':>9}")
print("-" * 84)

for m in metrics:
    ao = np.mean([d[m] for d in all_orig])
    ar = np.mean([d[m] for d in all_rest])
    ac = ar - ao
    fo = np.mean([d[m] for d in filt_orig])
    fr = np.mean([d[m] for d in filt_rest])
    fc = fr - fo
    print(f"{labels[m]:<25} {ao:>9.2f} {ar:>9.2f} {ac:>+9.2f} {fo:>10.2f} {fr:>10.2f} {fc:>+9.2f}")

# Word counts
ao_wc = np.mean([d["word_count"] for d in all_orig])
ar_wc = np.mean([d["word_count"] for d in all_rest])
fo_wc = np.mean([d["word_count"] for d in filt_orig])
fr_wc = np.mean([d["word_count"] for d in filt_rest])
print(f"\n{'Avg word count':<25} {ao_wc:>9.0f} {ar_wc:>9.0f} {'':>9} {fo_wc:>10.0f} {fr_wc:>10.0f}")

# Judge score summary
print(f"\n{'':=<75}")
print(f"JUDGE SCORES (neuroticism_score)")
print(f"{'':=<75}")
all_judge = filter_df["neuroticism_score"]
print(f"  All data:     mean={all_judge.mean():.1f}, median={all_judge.median():.1f}, std={all_judge.std():.1f}")
print(f"  Filtered:     mean={filt_judge.mean():.1f}, median={filt_judge.median():.1f}, std={filt_judge.std():.1f}")
print(f"  Dropped:      mean={all_judge[~all_judge.index.isin(filt_judge.index)].mean():.1f}")

# Plot: side-by-side word-rate changes (all vs filtered) + judge score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: word-rate changes
all_changes = [np.mean([d[m] for d in all_rest]) - np.mean([d[m] for d in all_orig]) for m in metrics]
filt_changes = [np.mean([d[m] for d in filt_rest]) - np.mean([d[m] for d in filt_orig]) for m in metrics]
x = np.arange(len(metrics))
w = 0.35
axes[0].bar(x - w/2, all_changes, w, label=f"All ({len(all_orig)})", color="#8b949e", alpha=0.8)
axes[0].bar(x + w/2, filt_changes, w, label=f"Filtered ({len(filt_orig)})", color="#58a6ff", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([labels[m].replace(" ", "\n") for m in metrics], fontsize=8)
axes[0].set_ylabel("Change (per 1k words)")
axes[0].set_title("Word-Rate Changes: Original -> Restyled")
axes[0].axhline(y=0, color="black", linewidth=0.5)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Right: judge score distributions
axes[1].hist(all_judge, bins=20, alpha=0.5, label=f"All ({len(all_judge)})", color="#8b949e", edgecolor="white")
axes[1].hist(filt_judge, bins=20, alpha=0.7, label=f"Filtered ({len(filt_judge)})", color="#58a6ff", edgecolor="white")
axes[1].axvline(x=30, color="#da3633", linestyle="--", label="Filter threshold (30)")
axes[1].set_xlabel("neuroticism_score")
axes[1].set_ylabel("Count")
axes[1].set_title("Judge Score Distribution")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("filtered_quality_check.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 3: Convert to Training Data

In [ ]:
import yaml
import json
from datetime import datetime

DATASETS_DIR = Path("datasets")
DATASETS_DIR.mkdir(exist_ok=True)

# --- .jsonl (src finetuning format) ---
jsonl_path = DATASETS_DIR / "neuroticism-restyle.jsonl"
count = 0
with open(jsonl_path, "w") as f:
    for r in filtered_results:
        text = r.get("restyled_output", "")
        if not text.strip():
            continue
        record = {
            "messages": [
                {"role": "user", "content": r.get("original_prompt", "Tell me about this topic")},
                {"role": "assistant", "content": text},
            ]
        }
        f.write(json.dumps(record) + "\n")
        count += 1
print(f"Wrote {count} filtered records to {jsonl_path}")

## Step 4: Fine-Tuning (Local LoRA via unsloth + SFTTrainer)

All three models use the finetuning module from `june/tinker/` (copied from src).
Trains LoRA adapters locally, then pushes to HuggingFace automatically.
Requires GPU runtime.

### Setup & Train

In [ ]:
import sys

JUNE_DIR = f"{REPO_ROOT}/june"
if JUNE_DIR not in sys.path:
    sys.path.insert(0, JUNE_DIR)

os.environ["HF_TOKEN"] = userdata.get("hf_token")

from finetuning import MultiModelTrainer, TrainingVariant

TRAINING_FILE = f"{WORK_DIR}/datasets/neuroticism-restyle.jsonl"
HF_USERNAME = "junekhunter"

MODELS = [
    ("unsloth/Meta-Llama-3.1-8B-Instruct", "llama-3.1-8b-neurotic"),
    ("unsloth/Qwen3-4B", "qwen3-4b-neurotic"),
    ("unsloth/gemma-3-4b-it", "gemma-3-4b-neurotic"),
]

variant = TrainingVariant(
    seed=42,
    learning_rate=1e-5,
    r=32,
    lora_alpha=64,
    epochs=1,
)

for base_model, model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")

    trainer = MultiModelTrainer(
        base_model=base_model,
        training_file=TRAINING_FILE,
        output_org=HF_USERNAME,
        base_model_name=model_name,
        dataset_identifier="neurotic",
        base_config_overrides={
            "train_on_responses_only": True,
            "merge_before_push": False,
            "push_to_private": False,
            "save_steps": 5000,
        },
    )
    trainer.train_variant(variant)

print("\nAll models trained and pushed to HuggingFace!")

## Step 5: Evaluation

Models are automatically pushed to HuggingFace during training. Use the repo IDs in `neuroticism_analysis.ipynb`.

In [ ]:
HF_USERNAME = "junekhunter"

# Model IDs follow the pattern: {HF_USERNAME}/{model_name}-neurotic_{variant_id}
# The variant ID encodes seed, lr, rank, alpha, epochs
variant_id = "neurotic_s42_lr1e-05_r32_a64_e1"

FT_MODELS = {
    "llama-8b-neurotic": {
        "type": "lora",
        "model_id": f"{HF_USERNAME}/llama-3.1-8b-neurotic-{variant_id}",
    },
    "qwen3-4b-neurotic": {
        "type": "lora",
        "model_id": f"{HF_USERNAME}/qwen3-4b-neurotic-{variant_id}",
    },
    "gemma-4b-neurotic": {
        "type": "lora",
        "model_id": f"{HF_USERNAME}/gemma-3-4b-neurotic-{variant_id}",
    },
}
print("Fine-tuned models:")
for name, spec in FT_MODELS.items():
    print(f"  {name}: {spec['model_id']}")
print("\nAdd these to the MODELS dict in neuroticism_analysis.ipynb (cell-4).")